In [1]:
%%capture
!pip install -q sentence-transformers pinecone

In [2]:
%%capture

import pandas as pd
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata   # Para usar los secrets de colab

In [3]:
df = pd.read_csv('/content/imdb_top_1000_fixed.csv')

# Crear el texto que se llevara a la base de datos
columns_to_combine = ['Overview', 'Director', 'Star1', 'Star2', 'Star3', 'Star4']
df['text'] = df[columns_to_combine].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)
df['ids'] = df.index.astype('str')

# Crea los embeddings de la columna text
embeddings = model.encode(df['text'], batch_size=64, show_progress_bar=True)
df['embeddings'] = embeddings.tolist()
df = df.fillna(' ')

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

## Pinecone

In [4]:
pincone_api = userdata.get('pincone_api')
pc = Pinecone(api_key=pincone_api)

db_name = 'movies-embeddings'
dimension_embeddings = len(df['embeddings'][0])

In [5]:
# Crear el índice si no existe
if db_name not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=db_name,
        dimension=dimension_embeddings,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"✅ Índice {db_name} creado exitosamente.")
else:
    print(f"ℹ️ El índice {db_name} ya existe.")

# Luego puedes conectarte al índice:
index = pc.Index(db_name)

ℹ️ El índice movies-embeddings ya existe.


In [6]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size=64

for i in tqdm(range(0, len(df), batch_size)):

    # find end of batch
    i_end = min(i+batch_size, len(df))
    # extract batch
    batch = df[i:i_end]
    # generate embeddings for batch
    ids = batch['ids']
    emb = batch['embeddings']
    metadata = batch.drop(['ids','embeddings','text'],axis=1).to_dict('records')

    # add all to upsert list
    to_upsert = list(zip(ids, emb, metadata))
    # upsert/insert these records to pinecone
    _ = index.upsert(to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/16 [00:00<?, ?it/s]

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1000}},
 'total_vector_count': 1000,
 'vector_type': 'dense'}

### Pinecone query

In [7]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [8]:
query = 'a history of time travel'
query_vector = model.encode(query).tolist()

responses = index.query(vector=query_vector, top_k = 3, include_metadata=True)

In [9]:
responses

{'matches': [{'id': '761',
              'metadata': {'Certificate': 'U',
                           'Director': 'Mamoru Hosoda',
                           'Genre': 'Animation, Adventure, Comedy',
                           'Gross': ' ',
                           'IMDB_Rating': 7.7,
                           'Meta_score': ' ',
                           'No_of_Votes': 60368.0,
                           'Overview': 'A high-school girl named Makoto '
                                       'acquires the power to travel back in '
                                       'time, and decides to use it for her '
                                       'own personal benefits. Little does she '
                                       'know that she is affecting the lives '
                                       'of others just as much as she is her '
                                       'own.',
                           'Poster_Link': 'https://m.media-amazon.com/images/M/MV5BMzA4ZGM1NjYtMjcxY

### Cargar índice de Pinecone

In [10]:
db_name = 'movies-embeddings'
index_2 = pc.Index(db_name)

In [11]:
query = 'a history of an space journey'
query_vector = model.encode(query).tolist()

responses = index_2.query(
  vector=query_vector,
  top_k=3,
  include_metadata=True,
)


In [12]:
responses

{'matches': [{'id': '686',
              'metadata': {'Certificate': 'PG',
                           'Director': 'Philip Kaufman',
                           'Genre': 'Adventure, Biography, Drama',
                           'Gross': '21,500,000',
                           'IMDB_Rating': 7.8,
                           'Meta_score': 91.0,
                           'No_of_Votes': 56235.0,
                           'Overview': 'The story of the original Mercury 7 '
                                       'astronauts and their macho, '
                                       'seat-of-the-pants approach to the '
                                       'space program.',
                           'Poster_Link': 'https://m.media-amazon.com/images/M/MV5BOTUwMDA3MTYtZjhjMi00ODFmLTg5ZTAtYzgwN2NlODgzMmUwXkEyXkFqcGdeQXVyNjc1NTYyMjg@._V1_UX67_CR0,0,67,98_AL_.jpg',
                           'Released_Year': '1983',
                           'Runtime': '193 min',
                           'Serie